# Import Library

In [58]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score as AS, recall_score as rs, precision_score as ps, classification_report as cr, f1_score as f1, confusion_matrix as cm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder as ohe, StandardScaler as ss
from sklearn.model_selection import train_test_split as tts, GridSearchCV as gscv

from sklearn.linear_model import LinearRegression, LinearRegression as lr, LogisticRegression
from sklearn.ensemble import RandomForestClassifier as rfc
from sklearn.neighbors import KNeighborsClassifier as knc
from sklearn.tree import DecisionTreeClassifier as dtc, plot_tree as pt, export_graphviz

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB as mnn

from sklearn.compose import ColumnTransformer as ct
import graphviz



# Load Data Set

In [16]:
dat = pd.read_csv(r"E:\DATA FOR TEST\Tele Customer churn\WA_Fn-UseC_-Telco-Customer-Churn.csv")
data = dat.copy()
print(data.info())
print(data.describe())
print(data.isnull().sum())
data.TotalCharges = pd.to_numeric(data.TotalCharges, errors = 'coerce')
data.dropna(how='any', inplace = True)
data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [63]:
x = data.drop(columns=['customerID', 'Churn'], axis = 1)

# Feature Encoding
x = pd.get_dummies(x, columns = ['gender', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'],
                   dtype = int, drop_first = True)

y = data.iloc[:, -1].values


# encoder = ct(transformers = [('encoder', ohe(), [:, :])], remainder = 'passthrough')
# x_train = encoder.fit_transform(x_train)
# x_test = encoder.transform(x_test)

# Divide data in Training and Testing
x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.2)

# KNeighborsClassifier

In [18]:
# By Pipeline
pipe = Pipeline([
    ('scaler', ss()),
    ('model', knc(n_neighbors=5))
])

pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print("Classification Report:\n", cr(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.87      0.84      1012
         Yes       0.61      0.53      0.56       395

    accuracy                           0.77      1407
   macro avg       0.72      0.70      0.70      1407
weighted avg       0.76      0.77      0.77      1407



In [59]:
# By Mannual
# Feature Sclaing
scaler = ss()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Importing K Neighbor Classifier
KnClassifier = knc(n_neighbors = 5)
KnClassifier.fit(x_train, y_train)
y_pred = KnClassifier.predict(x_test)

print(cr(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.81      0.85      0.83      1005
         Yes       0.57      0.50      0.54       402

    accuracy                           0.75      1407
   macro avg       0.69      0.68      0.68      1407
weighted avg       0.74      0.75      0.75      1407



In [20]:
# Find Best Neighbour Value
def find_neighbour_value(x_train, x_test, y_train, y_test):
    best_neighbour = None
    best_f1_score = -1   # important: start with lowest possible value

    scaler = ss()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    for neighbor in range(1, 100, 2):
        model = knc(n_neighbors=neighbor)
        model.fit(x_train_scaled, y_train)
        y_pred = model.predict(x_test_scaled)

        score = f1(y_test, y_pred, pos_label = 'Yes')

        if score > best_f1_score:
            best_f1_score = score
            best_neighbour = neighbor

    print("Best Neighbour Value:", best_neighbour)
    print("Best F1 Score:", best_f1_score)

find_neighbour_value(x_train, x_test, y_train, y_test)


Best Neighbour Value: 17
Best F1 Score: 0.5684830633284241


In [21]:
# Find Best Neighbour Value by Pipeline
pipe = Pipeline([
    ('scaler', ss()),
    ('model', knc())])

param_grid = {
    'model__n_neighbors': list(range(1, 100, 2))}

grid = gscv(
    pipe,
    param_grid=param_grid,
    scoring='f1',        # metric you care about
    cv=5,
    n_jobs=-1)

grid.fit(x_train, y_train)

y_pred = grid.predict(x_test)
print(cr(y_test, y_pred))

C:\Users\puruj\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


              precision    recall  f1-score   support

          No       0.81      0.81      0.81      1012
         Yes       0.52      0.52      0.52       395

    accuracy                           0.73      1407
   macro avg       0.67      0.67      0.67      1407
weighted avg       0.73      0.73      0.73      1407



# Decesion Tree

In [60]:
# Initilizing Model
Decesion_tree_model = dtc()

# Traing Model by Data
Decesion_tree_model.fit(x_train, y_train)

# Make Prediction by X Test Data
y_pred = Decesion_tree_model.predict(x_test)

# Show Classification Reports
print(cr(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.80      0.80      0.80      1005
         Yes       0.49      0.49      0.49       402

    accuracy                           0.71      1407
   macro avg       0.65      0.64      0.64      1407
weighted avg       0.71      0.71      0.71      1407



In [61]:
# Decision Tree By Pipeline
pipe = Pipeline([ ( 'model' , dtc() ) ])
pipe.fit(x_train, y_train)

# Prediction
y_pred = pipe.predict(x_test)

# Result
print("Results -->\n", cr(y_test, y_pred))

Results -->
               precision    recall  f1-score   support

          No       0.80      0.80      0.80      1005
         Yes       0.50      0.49      0.50       402

    accuracy                           0.71      1407
   macro avg       0.65      0.65      0.65      1407
weighted avg       0.71      0.71      0.71      1407



In [62]:
# Dicision Tree Visulization
plt.figure(figsize=(20, 16))
pt(
    Decesion_tree_model,
    filled=True,
    feature_names=x_train.columns,
    class_names=True
)
plt.savefig("decision_tree.jpg", dpi=600)
plt.show()

# # Changing Max Depth to 2 only
# Decesion_tree_model = dtc(max_depth = 2).fit(x_train, y_train)
# print(cr(y_test, Decesion_tree_model.predict(x_test)))

# # Dicision Tree Visulization
# plt.figure(figsize=(20, 16))
# pt(
#     Decesion_tree_model,
#     filled=True,
#     feature_names=x_train.columns,
#     class_names=True)
# plt.savefig("decision_tree1.jpg", dpi=200, bbox_inches="tight")
# plt.show()

AttributeError: 'numpy.ndarray' object has no attribute 'columns'

<Figure size 2000x1600 with 0 Axes>

In [51]:
# # Same Visulization With GraphViz
# export_graphviz(
#     Decesion_tree_model,
#     out_file="tree.jpg",
#     feature_names=x_train.columns,
#     class_names=True,
#     filled=True,
#     proportion=True,
#     special_characters=True
# )

# with open('tree.dot') as f:
#     dot_graph = f.read()

# graph = graphviz.Source(dot_graph, format='png')
# graph

# Random Forest

In [52]:
Random_Forest = rfc(n_estimators = 200)
Random_Forest.fit(x_train, y_train)
y_pred = Random_Forest.predict(x_test)
print("Results -->/n", cr(y_test, y_pred))

Results -->/n               precision    recall  f1-score   support

          No       0.80      0.91      0.85      1005
         Yes       0.66      0.45      0.53       402

    accuracy                           0.78      1407
   macro avg       0.73      0.68      0.69      1407
weighted avg       0.76      0.78      0.76      1407



In [53]:
# By Pipeline
pipe = Pipeline([
    ('model', rfc())])
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print("Results -->\n", cr(y_test, y_pred))

Results -->
               precision    recall  f1-score   support

          No       0.80      0.91      0.85      1005
         Yes       0.65      0.44      0.53       402

    accuracy                           0.77      1407
   macro avg       0.73      0.67      0.69      1407
weighted avg       0.76      0.77      0.76      1407



# Naive Bays 

In [54]:

# Create, train, predict with MultinomialNB
model = mnn()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

# Evaluate
print("Accuracy:", round(AS(y_test, y_pred)*100, 2))
print("\nClassification report:\n", cr(y_test, y_pred, target_names=["ham", "spam"]))

Accuracy: 67.8

Classification report:
               precision    recall  f1-score   support

         ham       0.88      0.64      0.74      1005
        spam       0.46      0.78      0.58       402

    accuracy                           0.68      1407
   macro avg       0.67      0.71      0.66      1407
weighted avg       0.76      0.68      0.69      1407



In [55]:
# Confusion Matrix
c_matrics = cm(y_test, y_pred)

TN, FP, FN, TP = c_matrics.ravel()
total = TN + FP + FN + TP

print("Confusion Matrix:")
print(c_matrics)
print("\nDetails (Counts and Percentages):")
print(f"True Negative (TN): {TN} ({TN/total*100:.2f}%)")
print(f"False Positive (FP): {FP} ({FP/total*100:.2f}%)")
print(f"False Negative (FN): {FN} ({FN/total*100:.2f}%)")
print(f"True Positive (TP): {TP} ({TP/total*100:.2f}%)")


Confusion Matrix:
[[639 366]
 [ 87 315]]

Details (Counts and Percentages):
True Negative (TN): 639 (45.42%)
False Positive (FP): 366 (26.01%)
False Negative (FN): 87 (6.18%)
True Positive (TP): 315 (22.39%)


# SVM

In [64]:
vector_model = SVC()
vector_model.fit(x_train, y_train)

y_pred = vector_model.predict(x_test)

print("Classification Reports :\n", cr(y_test, y_pred))

Classification Reports :
               precision    recall  f1-score   support

          No       0.74      1.00      0.85      1035
         Yes       0.00      0.00      0.00       372

    accuracy                           0.74      1407
   macro avg       0.37      0.50      0.42      1407
weighted avg       0.54      0.74      0.62      1407



C:\Users\puruj\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\puruj\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\puruj\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [65]:
# Confusion Matrix
c_matrics = cm(y_test, y_pred)

TN, FP, FN, TP = c_matrics.ravel()
total = TN + FP + FN + TP

print("Confusion Matrix:")
print(c_matrics)

print("\nDetails (Counts and Percentages):")
print(f"True Negative (TN): {TN} ({TN/total*100:.2f}%)")
print(f"False Positive (FP): {FP} ({FP/total*100:.2f}%)")
print(f"False Negative (FN): {FN} ({FN/total*100:.2f}%)")
print(f"True Positive (TP): {TP} ({TP/total*100:.2f}%)")


Confusion Matrix:
[[1035    0]
 [ 372    0]]

Details (Counts and Percentages):
True Negative (TN): 1035 (73.56%)
False Positive (FP): 0 (0.00%)
False Negative (FN): 372 (26.44%)
True Positive (TP): 0 (0.00%)


# Logistic Regression

In [119]:
data = {
    'Age': [22, 25, 18, 45, 12, 43, 23, 33],
    'Gender': ['F', 'F', 'M', 'M', 'F', 'M', 'M', 'M'],
    'Smoker': ['N', 'S', 'S', 'N', 'S', 'S', 'S', 'S'],
    'Disease': [1, 1, 0, 0, 0, 1, 0, 1]}

df = pd.DataFrame(data)
print(df)

   Age Gender Smoker  Disease
0   22      F      N        1
1   25      F      S        1
2   18      M      S        0
3   45      M      N        0
4   12      F      S        0
5   43      M      S        1
6   23      M      S        0
7   33      M      S        1


In [120]:
# Feature Encoding

# 1
numerical_data = df.select_dtypes(include = ['int', 'float'])
categorical_data = df.select_dtypes(include = ['object', 'category'])

col_feature_encoding = pd.get_dummies(categorical_data, dtype = int)
encoded_df = pd.concat([numerical_data, col_feature_encoding], axis = 1)
encoded_df

# 2
encoded_df = pd.get_dummies(df, dtype = int)
encoded_df

# 3 
encoded_df = pd.get_dummies(df, columns = ['Gender', 'Smoker'], dtype = int)
encoded_df

,Age,Disease,Gender_F,Gender_M,Smoker_N,Smoker_S
0,22,1,1,0,1,0
1,25,1,1,0,0,1
2,18,0,0,1,0,1
3,45,0,0,1,1,0
4,12,0,1,0,0,1
5,43,1,0,1,0,1
6,23,0,0,1,0,1
7,33,1,0,1,0,1


In [143]:
# Seperate X and Y Variabel
x = encoded_df.drop(columns = ['Disease'], axis=1)
y = encoded_df['Disease']

# Separate Traning Data and Testing Data
x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.5, random_state = 42, stratify=y)

# Feature Scaling
scaler = ss()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [144]:
# Model Initilization
logistic_model = LogisticRegression(max_iter = 1000)
logistic_model.fit(x_train, y_train)

y_pred = logistic_model.predict(x_test)
print("Classification Report :", cr(y_test, y_pred))

Classification Report :               precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4



In [146]:
# By Pipeline
pipe = Pipeline([
    ('scler', ss()),
    ('model', LogisticRegression())])
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print("Classification Report:", cr(y_test, y_pred))

Classification Report:               precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4

